# HR Attrition Analytics — Walkthrough
**Business question:** Why do employees leave, what does it cost, which groups are at highest risk, and is pay fair?

**Data:** IBM HR Analytics Employee Attrition sample — 1,470 employees, 35 attributes.
SQL (DuckDB) builds the clean model and KPI tables; Python (SciPy, scikit-learn, statsmodels) handles tests, segmentation and the pay-equity regression.

In [1]:
import duckdb, pandas as pd, json
pd.set_option('display.width', 180); pd.set_option('display.max_colwidth', 90)
con = duckdb.connect('../data/hr.duckdb', read_only=True)
q = lambda s: con.sql(s).df()
T = '../outputs/tables/'

## 1. Data quality and KPIs

In [2]:
display(q('SELECT * FROM data_quality_log'))
q('SELECT * FROM headline_kpis').T

,check_name,value
0,Employees,1470
1,Duplicate employee IDs,0
2,Null values (all columns),0
3,Constant columns dropped,"EmployeeCount, Over18, StandardHours"


,0
employees,1470.00
leavers,237.00
attrition_rate_pct,16.10
est_replacement_cost,6807246.00
avg_income_leavers,4787.00
avg_income_stayers,6833.00
avg_tenure_leavers,5.10
avg_tenure_stayers,7.40
first_year_attrition_pct,34.90
avg_engagement,2.73


Replacement cost assumes **50% of annual salary per leaver**, the conservative end of the commonly cited 50–200% range.

## 2. Attrition drivers

In [3]:
q("SELECT * FROM attrition_by_dimension WHERE dimension IN ('Overtime','Tenure band','Income band','Stock option level','Marital status','Business travel')")

,dimension,value,employees,leavers,attrition_rate_pct,share_of_leavers_pct,est_replacement_cost
0,Business travel,Travel_Frequently,277,69.0,24.9,29.1,1772514.0
1,Business travel,Travel_Rarely,1043,156.0,15.0,65.8,4657398.0
2,Business travel,Non-Travel,150,12.0,8.0,5.1,377334.0
3,Income band,< 3K,395,113.0,28.6,47.7,1553736.0
4,Income band,3K-6K,519,66.0,12.7,27.8,1743132.0
5,Income band,6K-10K,275,33.0,12.0,13.9,1570956.0
6,Income band,10K+,281,25.0,8.9,10.5,1939422.0
7,Marital status,Single,470,120.0,25.5,50.6,3164952.0
8,Marital status,Married,673,84.0,12.5,35.4,2801646.0
9,Marital status,Divorced,327,33.0,10.1,13.9,840648.0


![drivers](../outputs/charts/01_attrition_by_driver.png)

In [4]:
q('SELECT * FROM overtime_level_matrix')

,job_level,overtime,employees,attrition_rate_pct
0,1,No overtime,387,15.8
1,1,Overtime,156,52.6
2,2,No overtime,388,6.7
3,2,Overtime,146,17.8
4,3,No overtime,155,12.3
5,3,Overtime,63,20.6
6,4,No overtime,73,2.7
7,4,Overtime,33,9.1
8,5,No overtime,51,3.9
9,5,Overtime,18,16.7


![overtime](../outputs/charts/02_overtime_by_level.png)

In [5]:
q('SELECT * FROM role_scorecard')

,department,job_role,headcount,leavers,attrition_rate_pct,overtime_pct,median_monthly_income,avg_engagement,avg_years_since_promotion,est_replacement_cost,risk_rank
0,Sales,Sales Representative,83,33.0,39.8,29.0,2579.0,2.72,1.1,468216.0,1
1,Research & Development,Laboratory Technician,259,62.0,23.9,24.0,2886.0,2.70,1.4,1085964.0,2
2,Human Resources,Human Resources,52,12.0,23.1,25.0,3093.0,2.75,1.3,267534.0,3
3,Sales,Sales Executive,326,57.0,17.5,29.0,6231.0,2.73,2.5,2561238.0,4
4,Research & Development,Research Scientist,292,47.0,16.1,33.0,2888.0,2.74,1.5,784092.0,5
5,Research & Development,Manufacturing Director,145,10.0,6.9,27.0,6447.0,2.75,2.1,441930.0,6
6,Research & Development,Healthcare Representative,131,9.0,6.9,28.0,6811.0,2.74,3.0,461604.0,7
7,Research & Development,Manager,54,3.0,5.6,24.0,17546.0,2.77,5.5,271908.0,8
8,Sales,Manager,37,2.0,5.4,27.0,17048.0,2.74,4.1,232014.0,9
9,Research & Development,Research Director,80,2.0,2.5,29.0,16510.0,2.70,3.2,232746.0,10


![roles](../outputs/charts/03_attrition_by_role.png)
![sat](../outputs/charts/04_satisfaction.png)

In [6]:
pd.read_csv(T + 'leaver_vs_stayer_profile.csv')

,metric,Leavers,Stayers
0,years_at_company,5.10,7.40
1,total_working_years,8.20,11.90
2,distance_km,10.60,8.90
3,work_life_balance,2.66,2.78
4,job_satisfaction,2.47,2.78
5,engagement_index,2.54,2.77
6,env_satisfaction,2.46,2.77
7,age,33.60,37.60
8,monthly_income,4787.00,6833.00
9,overtime_pct,53.60,23.40


## 3. Which differences are statistically significant?

In [7]:
pd.read_csv(T + 'driver_significance_tests.csv')

,feature,test,p_value,effect_size,median_leavers,median_stayers,significant_at_5pct
0,total_working_years,Mann-Whitney U,2.400000e-14,0.312,7.0,10.0,True
1,monthly_income,Mann-Whitney U,2.950000e-14,0.311,3202.0,5204.0,True
2,years_at_company,Mann-Whitney U,2.920000e-13,0.298,3.0,6.0,True
3,years_in_role,Mann-Whitney U,4.430000e-12,0.280,2.0,3.0,True
4,years_with_manager,Mann-Whitney U,1.810000e-11,0.272,2.0,3.0,True
5,age,Mann-Whitney U,5.300000e-11,0.269,32.0,36.0,True
6,overtime,chi-square,8.160000e-21,0.244,NaN,NaN,True
7,job_role,chi-square,2.750000e-15,0.242,NaN,NaN,True
8,job_level,chi-square,6.630000e-15,0.222,NaN,NaN,True
9,stock_option_level,chi-square,4.380000e-13,0.203,NaN,NaN,True


## 4. Explainable flight-risk segments
A shallow decision tree gives rules HR can act on. Logistic regression is the accuracy benchmark.

In [8]:
display(pd.read_csv(T + 'model_performance.csv'))
pd.read_csv(T + 'flight_risk_segments.csv')

,model,cv_auc,test_auc
0,"Decision tree (depth 3, explainable rules)",0.686,0.692
1,Logistic regression (all drivers),0.837,0.804


,employees,leavers,attrition_rate_pct,rule
0,123,64,52.0,years_at_company <= 2.5 AND age <= 39.5 AND stock_option_level <= 0.5
1,91,42,46.2,years_at_company > 2.5 AND overtime AND job_level <= 1.5
2,117,27,23.1,years_at_company <= 2.5 AND age <= 39.5 AND stock_option_level > 0.5
3,45,10,22.2,years_at_company > 2.5 AND NOT overtime AND work_life_balance <= 1.5
4,41,6,14.6,years_at_company <= 2.5 AND age > 39.5 AND distance_km > 8.5
5,221,32,14.5,years_at_company > 2.5 AND overtime AND job_level > 1.5
6,61,5,8.2,years_at_company <= 2.5 AND age > 39.5 AND distance_km <= 8.5
7,771,51,6.6,years_at_company > 2.5 AND NOT overtime AND work_life_balance > 1.5


In [9]:
print(open(T + 'decision_tree_rules.txt').read())

|--- years_at_company <= 2.5
|   |--- age <= 39.5
|   |   |--- stock_option_level <= 0.5
|   |   |   |--- class: 1
|   |   |--- stock_option_level >  0.5
|   |   |   |--- class: 1
|   |--- age >  39.5
|   |   |--- distance_km <= 8.5
|   |   |   |--- class: 0
|   |   |--- distance_km >  8.5
|   |   |   |--- class: 0
|--- years_at_company >  2.5
|   |--- overtime <= 0.5
|   |   |--- work_life_balance <= 1.5
|   |   |   |--- class: 1
|   |   |--- work_life_balance >  1.5
|   |   |   |--- class: 0
|   |--- overtime >  0.5
|   |   |--- job_level <= 1.5
|   |   |   |--- class: 1
|   |   |--- job_level >  1.5
|   |   |   |--- class: 0



## 5. What-if and pay equity

In [10]:
s = json.loads(open(T + 'analysis_summary.json').read())
s

{'tree_test_auc': 0.692,
 'lr_test_auc': 0.804,
 'whatif_overtime': {'leavers_avoided': 83.0,
  'cost_avoided': 2278126.0,
  'pct_of_all_leavers': 35.1},
 'pay_equity': {'raw_median_gap_pct': -5.0,
  'adjusted_gap_pct': 0.01,
  'p_value': 0.994,
  'r_squared': 0.87}}

## 6. Recommendations
1. **Cap and monitor overtime for job-level-1 staff.** Their attrition is 53% with overtime vs 16% without; this lever alone is worth ≈83 avoided exits.
2. **Early-career retention package.** For employees under 40 in their first 3 years, extend stock-option grants (no-stock attrition is 24% vs 8–9% with level 1–2 grants).
3. **Review pay for Sales Representatives and Laboratory Technicians.** Both have median pay below $3K a month and attrition of 40% and 24%.
4. **Act on survey signals.** Employees who score work-life balance or environment satisfaction at 1 are early warnings and should get a manager follow-up.
5. **Pay equity:** no significant adjusted gender gap. Keep running this check every review cycle.